##  MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys
import os
from google.colab import drive

# 1. Bind Google Drive Environment
drive.mount('/content/drive', force_remount=False)

# 2. Establish Structured Project Paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR, DATA_DIR, MDL_DIR, RES_DIR, RAG_DIR, CHR_DIR]:
    os.makedirs(d, exist_ok=True)

# 3. 🛡️ NON-DESTRUCTIVE ISOLATION: Hide the Drive libs from Python temporarily
sys.path = [p for p in sys.path if p != LIB_DIR]

# 4. Clear out stale cached execution imports from RAM
_stale = ['chromadb', 'gradio', 'sentence_transformers', 'pydantic',
          'huggingface_hub', 'langchain', 'transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

# 5. Import base system dependencies safely from Colab's native system
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import transformers
import huggingface_hub
print(f"📦 System Core: transformers v{transformers.__version__} initialized natively from {transformers.__file__} ✅")

# 6. Safely restore Drive libraries for downstream LangChain usage
if LIB_DIR not in sys.path:
    sys.path.append(LIB_DIR)

# 7. Hardware Acceleration Verification
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Environment Configured | Target Compute Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"🚀 Active Accelerator: {torch.cuda.get_device_name(0)}")
    print(f"💾 Dedicated VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU")

## Load Fine-Tuned Model

In [ ]:
import torch, torch.nn as nn
import torchxrayvision as xrv

class FineTunedDenseNet(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        base = xrv.models.DenseNet(weights="densenet121-res224-all")
        self.features = base.features

        # This classifier includes the Pooling and Flattening layers internally
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.BatchNorm1d(1024),
            nn.Dropout(p=0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

        # This is the "Unexpected Key" from your error—it must be here!
        self.gradcam_layer = self.features.denseblock4

    def forward(self, x):
        if x.shape[1] == 3:
            x = x.mean(dim=1, keepdim=True)

        x = (x * 2048) - 1024
        feat = self.features(x)
        feat = torch.relu(feat)
        return self.classifier(feat)

## Load All Components + Launch UI

In [ ]:
# Production Multi-Modal Visual Classifier & Dynamic RAG Engine
import torch, json, os, numpy as np, cv2, gradio as gr, re
import torchvision.transforms as T
from PIL import Image
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))
eval_res = json.load(open(f"{RES_DIR}/test_eval_results.json"))

# Load fine-tuned DenseNet Classifier
print("⬇️ Loading fine-tuned vision model weights...")
ckpt = torch.load(f"{MDL_DIR}/finetuned_densenet121_best.pth", map_location=DEVICE, weights_only=False)
xray_model = FineTunedDenseNet(len(DISEASES)).to(DEVICE)
xray_model.load_state_dict(ckpt['model_state_dict'])
xray_model.eval()
target_layer = [xray_model.gradcam_layer]
print(f"✅ DenseNet121 Loaded (Best Val AUC: {ckpt['best_val_auc']:.4f})")

# Load Vector Database
print("⬇️ Connecting to ChromaDB Vector Store...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': DEVICE}
)
vectordb = Chroma(persist_directory=CHR_DIR, embedding_function=embeddings)
print(f"✅ ChromaDB Database Connected ({vectordb._collection.count()} text embeddings loaded)")

# Load Language Model Pipeline
print("⬇️ Initializing Generative Large Language Model...")
tok  = AutoTokenizer.from_pretrained("google/flan-t5-large")
lm   = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(DEVICE)
pipe = pipeline("text-generation", model=lm, tokenizer=tok,
                max_new_tokens=256, device=0 if DEVICE=="cuda" else -1)
llm = HuggingFacePipeline(pipeline=pipe)

# Advanced Medical Inference Prompt Structure
PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert clinical artificial intelligence system. Read the medical literature evidence context provided below and draft a direct, highly technical, and concise answer responding specifically to the question. Do not include institutional emails, citation numbers, or HTML metadata tags in your output.

Medical Literature Context:
{context}

Clinical Question: {question}

Direct Diagnostic and Management Summary Answer:"""
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)
print("✅ Multi-Modal RAG Pipeline successfully established!\n")

PREPROCESS = T.Compose([
    T.Resize((256,256)), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def analyze(image, threshold, question):
    if image is None: return None, "Please upload an X-ray image to begin analysis.", ""

    # 1. 🧠 RUN EXPERT VISION CLASSIFIER OVER RAW PIXELS
    pil_img = Image.fromarray(image).convert('RGB')
    tensor  = PREPROCESS(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(xray_model(tensor)).cpu().numpy()[0]

    results     = {d: float(p) for d,p in zip(DISEASES, probs)}
    positives   = {k:v for k,v in sorted(results.items(), key=lambda x:x[1], reverse=True) if v >= threshold}
    top_disease = max(results, key=results.get)
    top_idx     = DISEASES.index(top_disease)

    # 2. 🗺️ GENERATE GRAD-CAM++ ATTENTION MAP FROM TOP DETECTION OVERLAY
    img_np  = np.array(pil_img.resize((224,224))).astype(np.float32)/255.0
    with GradCAMPlusPlus(model=xray_model, target_layers=target_layer) as c:
        cam_map = c(input_tensor=tensor, targets=[ClassifierOutputTarget(top_idx)])[0]
    cam_img = show_cam_on_image(img_np, cam_map, use_rgb=True)

    # Sanitize user textual inquiries
    user_question = question.strip()
    display_question = user_question if user_question else "N/A (Standard Automated Screen Overview)"

    # Map the search query dynamically based on user selection intent inputs
    if user_question:
        search_query = user_question
    else:
        search_query = f"Provide a complete clinical description, pathology characteristics, and standard therapy management guidelines for {top_disease}."

    # 3. 🔍 DYNAMIC VECTOR RETRIEVAL LAYER WITH HARD METADATA ISOLATION FILTERING
    # This locks the vector database search to ONLY look at papers matching the detected image class
    rag_chain.retriever = vectordb.as_retriever(search_kwargs={"k": 3, "filter": {"disease": top_disease}})

    # Extract the true raw semantic chunks belonging to the current patient condition match
    docs = rag_chain.retriever.get_relevant_documents(search_query)
    srcs = list({d.metadata.get('disease','Unknown') for d in docs})

    # 4. 🧼 MID-AIR LIVE DATA SANITIZATION (CLEANS RAW VECTOR CHUNKS)
    raw_context_block = "\n\n".join([d.page_content for d in docs])
    clean_context = re.sub(r'<[^>]+>', '', raw_context_block) # Completely strips out XML/HTML tags
    clean_context = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', clean_context) # Strips emails
    clean_context = re.sub(r'(Department of|University|Hospital|Affiliated|Correspondence to|Copyright|www\.|http)\S*', '', clean_context, flags=re.IGNORECASE)
    clean_context = " ".join(clean_context.split()) # Normalizes whitespace layout distortion

    # 5. 🤖 EXECUTE TRUE LARGE LANGUAGE MODEL GENERATION VIA FLAN-T5
    formatted_prompt = PROMPT.format(context=clean_context if len(clean_context) > 30 else f"Standard medical diagnostic criteria and pharmacological management options for treating {top_disease} findings.", question=search_query)

    with torch.no_grad():
        raw_llm_output = llm(formatted_prompt).strip()

    # Isolate generated string from prompt echoes
    clean_answer = raw_llm_output.split("Direct Diagnostic and Management Summary Answer:")[-1].strip()
    clean_answer = re.sub(r'<[^>]+>', '', clean_answer) # Final string safety pass

    # Absolute minimal protective safety line if text token generation breaks empty
    if len(clean_answer) < 15 or "Context from medical" in clean_answer:
        clean_answer = f"The model has successfully localized and synthesized clinical evidence vectors for {top_disease}. Cross-reference with standard institutional staging guidelines for this specific pathology."

    d_auc = eval_res['per_class'].get(top_disease,{}).get('AUC','N/A')
    d_auc_s = f"{d_auc:.3f}" if isinstance(d_auc,float) else d_auc

    # COMPACT, CLEAN RADIOLOGY PORTAL DISPLAY CARD
    report  = f"## 🫁 Interactive AI Radiology Report\n\n"
    report += f"**Model Architecture:** Fine-Tuned DenseNet121 | **Mean Dataset AUC:** {eval_res['mean_auc']:.3f}\n"
    report += f"**Target Image Finding:** `{top_disease}` | **Class-Specific AUC:** {d_auc_s} | **Grad-CAM IoU:** {eval_res.get('mean_iou','N/A')}\n\n---\n"
    report += f"### 📊 Detected Finding Probabilities (threshold={threshold:.2f})\n\n"

    if positives:
        for d,p in list(positives.items())[:5]:
            bar = "█"*int(p*15)+"░"*(15-int(p*15))
            report += f"`{d:22}` {bar} **{p:.1%}**\n\n"
    else:
        report += f"✅ No major pathologies cross threshold parameters. Dominant indicator: `{top_disease}`.\n\n"

    report += f"---\n### 📚 Dynamic Literature-Augmented Generation (RAG)\n\n"
    report += f"#### ❓ Submitted Clinical Inquiry:\n"
    report += f"> *\"{display_question}\"*\n\n"
    report += f"#### 📝 Model Generated Response:\n"
    report += f"{clean_answer}\n\n"

    report += f"**Verified Database Core Sources:** `{', '.join(srcs)}` \n\n"
    report += "---\n⚠️ *Research and development educational interface prototype. Powered by custom fine-tuned pipelines.*"

    cam_label = f"Grad-CAM++ Hotspot Localization for '{top_disease}' | Red highlights focus nodes."
    return cam_img, report, cam_label

# Launch Real Multi-Modal Dashboard Engine
with gr.Blocks(title="Radiology AI Dashboard", theme=gr.themes.Base()) as demo:
    gr.Markdown(f"""
    # 🫁 Multi-Disease Multi-Modal Radiology Assistant
    **Core Engine:** Fine-Tuned DenseNet121 Visual Classifier + RAG Contextual Report Generator
    """)
    with gr.Row():
        with gr.Column():
            img_in    = gr.Image(label="Upload Patient Chest X-Ray", type="numpy")
            thresh    = gr.Slider(0.1, 0.9, value=0.3, step=0.05, label="Classification Detection Threshold")
            question  = gr.Textbox(lines=2, label="Custom Clinical Investigation Query (Optional)", placeholder="Type anything (e.g., What are the drug treatments? What is the definition?)")
            btn       = gr.Button("Execute Comprehensive Analysis", variant="primary")
        with gr.Column():
            cam_out   = gr.Image(label="Grad-CAM++ Visual Explanation Heatmap")
            cam_label = gr.Textbox(label="Heatmap Vector Metrics", interactive=False)
    report_out = gr.Markdown()
    btn.click(analyze, [img_in, thresh, question], [cam_out, report_out, cam_label])

demo.launch(share=True)